# 02 — Data quality checks

This notebook checks whether the IST data are consistent with the documentation (see `docs/variables.md`, column *Allowed values*).

**Input:** `data/interim/IST_corrected_utf8.csv`, produced by `01_data_loading.ipynb` (raw file converted from Windows-1250 to UTF-8).

**Checks**
1. Categorical variables: allowed values
2. Numeric variables: ranges
3. Investigation of the surprises found in 1–2
4. Pilot phase, reconstructed from `RDATE`
5. Duplicate rows

The findings are summarised at the end. The resulting cleaning decisions are listed in `docs/variables.md`.

## Setup

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

UTF8_PATH = Path("../data/interim/IST_corrected_utf8.csv")

con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE VIEW ist AS
    SELECT * FROM read_csv_auto('{UTF8_PATH.as_posix()}')
""")

In [ ]:
con.sql("SELECT COUNT(*) AS n_patients FROM ist").df()

Expected: 19,435 patients. Each row is a patient; the dataset has no patient identifier.

## 1. Categorical variables: allowed values

For each variable: its values, with number and percentage of patients. Missing values appear as `None` / `NaN`.

In [ ]:
categorical = [
    "RCONSC",
    "SEX",
    "RSLEEP",
    "RATRIAL",
    "RCT",
    "RVISINF",
    "RHEP24",
    "RASP3",
    "RDEF1",
    "RDEF2",
    "RDEF3",
    "RDEF4",
    "RDEF5",
    "RDEF6",
    "RDEF7",
    "RDEF8",
    "STYPE",
    "RXASP",
    "RXHEP",
    "OCCODE",
    "ID14",
    "FDEAD",
    "ISC14",
    "PE14",
    "H14",
    "TRAN14",
    "SET14D",
]

for variable in categorical:
    display(
        con.sql(f"""
        SELECT {variable},
               COUNT(*) AS n_patients,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
        FROM ist
        GROUP BY {variable}
        ORDER BY {variable}
    """).df()
    )

**Findings**

- `RCONSC`, `SEX`, `RSLEEP`, `RCT`, `STYPE`, `ID14`: only allowed values, no missing values. The "other" stroke subtype is coded `OTH`.
- `RXASP`: 50% / 50%, as expected from a 1:1 randomisation.
- `RXHEP`: contains `H` (245 patients), the pilot-phase code for the medium dose (see 4.3).
- `RDEF1`–`RDEF8`: the value `C` (can't assess) is frequent for some deficits (≈20% for `RDEF5`, ≈18% for `RDEF6`).
- Missing values: `RATRIAL` and `RASP3` 984 each (see 3.1), `RHEP24` 344 (see 3.5 and 4.4).
- `RVISINF`: no missing values, although 6,411 patients had no CT (see 3.2).
- `OCCODE`: value `0` (97 patients) is not in the dictionary; no `8`; `9` for 53 patients (see 3.3).
- `FDEAD`: besides Y/N, the value `U` (57 patients) and missing values (99) (see 3.4).
- `SET14D` = 0 for 2 patients: for them, `ID14` = 0 means "unknown", not "alive".
- Benefit and safety outcomes (`ISC14`, `PE14`, `H14`, `TRAN14`) are rare, all below 2%.

## 2. Numeric variables: ranges

In [ ]:
numeric = ["AGE", "RDELAY", "RSBP"]

summaries = [
    con.sql(f"""
        SELECT '{variable}' AS variable,
               MIN({variable}) AS min,
               MAX({variable}) AS max,
               COUNT(*) - COUNT({variable}) AS n_missing
        FROM ist
    """).df()
    for variable in numeric
]
pd.concat(summaries, ignore_index=True)

Distribution of `AGE` at the two extremes:

In [ ]:
con.sql("""
    SELECT AGE, COUNT(*) AS n_patients
    FROM ist
    GROUP BY AGE
    ORDER BY AGE
    LIMIT 10
""").df()

In [ ]:
con.sql("""
    SELECT AGE, COUNT(*) AS n_patients
    FROM ist
    GROUP BY AGE
    ORDER BY AGE DESC
    LIMIT 10
""").df()

**Findings**

- `RDELAY` 1–48 h, `RSBP` 70–295 mmHg, `AGE` 16–99 years: all within plausible ranges, no missing values. `RDELAY` respects the 48 h inclusion criterion.
- `AGE`: 2 patients aged 16. The counts decrease smoothly up to 99 (1 patient): 99 is a real age, not a truncation code.

## 3. Investigations

### 3.1 Are the patients without `RATRIAL` the same as those without `RASP3`?

In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_both_missing
    FROM ist
    WHERE RATRIAL IS NULL AND RASP3 IS NULL
""").df()

984: the same patients. `RATRIAL` is not recorded in the pilot phase (dictionary); `RASP3` probably was not either, although the dictionary does not say so (confirmed in section 4).

### 3.2 `RVISINF` (infarct visible on CT) for patients without CT

In [ ]:
con.sql("""
    SELECT RCT, RVISINF, COUNT(*) AS n_patients
    FROM ist
    GROUP BY RCT, RVISINF
    ORDER BY RCT, RVISINF
""").df()

- 6,410 patients without CT have `RVISINF` = N: here N means "no CT", not "no visible infarct". `RVISINF` = N therefore mixes two different situations.
- 1 patient has `RCT` = N and `RVISINF` = Y, an impossible combination: likely a recording error.

### 3.3 What does `OCCODE` = 0 mean?

In [ ]:
con.sql("""
    SELECT FDEAD, COUNT(*) AS n_patients
    FROM ist
    WHERE OCCODE = 0
    GROUP BY FDEAD
    ORDER BY FDEAD
""").df()

All 97 patients with `OCCODE` = 0 have `FDEAD` missing: 0 means "6-month outcome missing", like 8 and 9. Missing 6-month outcome: 97 + 53 = 150 patients (0.8%), consistent with the 99% complete follow-up reported in the paper.

### 3.4 Agreement between `OCCODE` and `FDEAD` (death at 6 months)

In [ ]:
con.sql("""
    SELECT OCCODE, FDEAD, COUNT(*) AS n_patients
    FROM ist
    GROUP BY OCCODE, FDEAD
    ORDER BY OCCODE, FDEAD
""").df()

- All patients with `OCCODE` = 1 (dead) have `FDEAD` = Y (4,241).
- 128 patients have `OCCODE` = 2 (dependent) but `FDEAD` = Y: most likely deaths after 6 months (the dictionary warns that `FDEADD` is not necessarily within 6 months). The opposite disagreement (dead for `OCCODE`, alive for `FDEAD`) never occurs.
- `FDEAD` = U means unknown: 53 of its 57 patients have `OCCODE` = 9.
- 6 patients have a 6-month outcome in `OCCODE` (2–4) but `FDEAD` = U or missing: minor inconsistencies.

→ Use `OCCODE` for the 6-month status, not `FDEAD`.

### 3.5 Who are the patients without `RHEP24`?

In [ ]:
con.sql("""
    SELECT RHEP24 IS NULL AS rhep24_missing,
           RATRIAL IS NULL AS ratrial_missing,
           COUNT(*) AS n_patients
    FROM ist
    GROUP BY rhep24_missing, ratrial_missing
    ORDER BY rhep24_missing, ratrial_missing
""").df()

All 344 patients without `RHEP24` also lack `RATRIAL`: they are pilot-phase patients (640 + 344 = 984). When the question was introduced is checked in 4.4.

## 4. Pilot phase from `RDATE`

`RDATE` is month-year text with Polish month abbreviations (`sty` = Jan, `lut` = Feb, …, `paź` = Oct, `gru` = Dec), e.g. `paź-91` = October 1991. As text it does not sort chronologically, so we define two small SQL **macros** (reusable expressions with a name, like functions) that extract the year and the month number.

In [ ]:
con.sql("""
    CREATE OR REPLACE MACRO rdate_year(d) AS
        1900 + CAST(right(d, 2) AS INTEGER)
""")
con.sql("""
    CREATE OR REPLACE MACRO rdate_month(d) AS
        list_position(
            ['sty', 'lut', 'mar', 'kwi', 'maj', 'cze',
             'lip', 'sie', 'wrz', 'paź', 'lis', 'gru'],
            left(d, 3)
        )
""")

### 4.1 Patients per month, and patients without `RATRIAL`

In [ ]:
con.sql("""
    SELECT RDATE,
           rdate_year(RDATE) AS year,
           rdate_month(RDATE) AS month,
           COUNT(*) AS n_patients,
           COUNT(*) - COUNT(RATRIAL) AS without_ratrial
    FROM ist
    GROUP BY RDATE, year, month
    ORDER BY year, month
""").df()

- 65 months, from January 1991 to May 1996, with no gaps.
- Recruitment is low in 1991–1992 (4–79 patients per month), rises during 1993 and is stable at 450–680 per month from 1994: the transition from the pilot to the main phase.
- All patients up to February 1993 lack `RATRIAL`; in March 1993 only 1 of 180; none afterwards.

### 4.2 Size of the pilot phase

In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_until_feb93,
           COUNT(*) - COUNT(RATRIAL) AS without_ratrial
    FROM ist
    WHERE rdate_year(RDATE) * 100 + rdate_month(RDATE) <= 199302
""").df()

983 patients up to February 1993, all without `RATRIAL`, plus 1 in March 1993 = 984. **Pilot phase: January 1991 – early March 1993, 984 patients, identified exactly by `RATRIAL` missing.** `RDATE` alone cannot separate the two phases, because March 1993 contains patients of both.

### 4.3 Heparin coding in the two phases

In [ ]:
con.sql("""
    SELECT RATRIAL IS NULL AS pilot_phase,
           RXHEP,
           COUNT(*) AS n_patients
    FROM ist
    GROUP BY pilot_phase, RXHEP
    ORDER BY pilot_phase, RXHEP
""").df()

`H` appears only in the pilot phase and `M` only in the main phase. In the pilot phase, H (245) ≈ L (246) ≈ N / 2 (493), as in the trial design: `H` is the medium dose. → Recode H → M.

### 4.4 When was `RHEP24` introduced?

In [ ]:
con.sql("""
    SELECT RDATE,
           COUNT(*) AS n_patients,
           COUNT(*) - COUNT(RHEP24) AS without_rhep24
    FROM ist
    WHERE rdate_year(RDATE) * 100 + rdate_month(RDATE) <= 199303
    GROUP BY RDATE
    ORDER BY rdate_year(RDATE), rdate_month(RDATE)
""").df()

`RHEP24` is missing for all patients from January 1991 to January 1992, for 33 of 36 in February 1992, and for none from March 1992: the question was added to the form in February 1992 (276 + 35 + 33 = 344).

## 5. Duplicate rows

In [ ]:
con.sql("""
    SELECT COUNT(*) AS n_distinct_rows,
           (SELECT COUNT(*) FROM ist) AS n_total_rows,
           (SELECT COUNT(*) FROM ist) - COUNT(*) AS n_duplicates
    FROM (
        SELECT DISTINCT * FROM ist
    )
""").df()

No exact duplicate rows.

## Summary

| Finding | Implication for cleaning |
|---|---|
| Pilot phase = 984 patients (Jan 1991 – early Mar 1993), identified by `RATRIAL` missing | create `pilot_phase` |
| `RATRIAL`, `RASP3` not recorded in the pilot phase | missing by design, decide how to handle |
| `RHEP24` missing for the 344 patients randomised before Feb 1992 | missing by design, decide how to handle |
| `RXHEP` = H is the pilot-phase code for the medium dose | recode H → M |
| `RVISINF` = N also for patients without CT; 1 impossible combination | combine with `RCT`; handle the anomaly |
| `RDEF1`–`RDEF8` = C (can't assess), up to ≈20% | decide how to encode |
| `OCCODE` = 0, 8, 9 are missing (150 patients) | set to missing |
| `FDEAD` may include deaths after 6 months | use `OCCODE` for the 6-month status |
| `SET14D` = 0 for 2 patients | their `ID14` is unknown |
| No duplicates; numeric variables within plausible ranges | — |

Cleaning decisions: see `docs/variables.md`, section *Cleaning decisions*.